# Week 6: The Price Is Right

Product price prediction from descriptions. Goal: beat the ~$76 baseline (NLP + LR).

**Order of play:** Data → Baseline → Fine-tune → Evaluate

**Setup:** Add `OPENAI_API_KEY` and `HF_TOKEN` to `.env`

**Run from:** `week6/` directory (or add path so `pricer` imports work)

In [2]:
import os
import sys
import re
import json
from pathlib import Path

from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI

# Add week6 to path so we can import pricer
week6_root = Path.cwd() if (Path.cwd() / 'pricer').exists() else Path.cwd().parent.parent
sys.path.insert(0, str(week6_root))

from pricer.items import Item
from pricer.evaluator import evaluate

load_dotenv(override=True)
hf_token = os.environ.get("HF_TOKEN", "")
login(hf_token, add_to_git_credential=True)
openai = OpenAI()

Token has not been saved to git credential helper.
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


## Load data from HuggingFace Hub

In [3]:
LITE_MODE = True  

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} train, {len(val):,} val, {len(test):,} test")

Loaded 20,000 train, 1,000 val, 1,000 test


## Baseline predictor (zero-shot)

Optional: test a simple zero-shot LLM predictor before fine-tuning.

In [4]:
def baseline_pricer(item):
    """Zero-shot price predictor using GPT-4o-mini. Returns price string for evaluate()."""
    prompt = f"Estimate the price of this product in USD. Respond with only the price, no explanation.\n\n{item.summary}"
    r = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50,
    )
    return r.choices[0].message.content.strip()

# Uncomment to run baseline (uses API calls)
evaluate(baseline_pricer, test, size=50)

  0%|          | 0/50 [00:00<?, ?it/s]

$19 $34 $29 $20 $50 $180 $78 $95 $0 $320 $113 $29 $10 $24 $29 $3 $41 $10 $90 $31 $34 $4 $5 $75 $83 $253 $105 $6 $51 $64 $30 $15 $40 $55 $35 $219 $15 $35 $44 $22 $154 $50 $20 $45 $120 $1 $12 $4 $55 $82 

## Prepare JSONL for fine-tuning

In [5]:
def messages_for(item):
    msg = f"Estimate the price of this product. Respond with the price, no explanation.\n\n{item.summary}"
    return [
        {"role": "user", "content": msg},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

def make_jsonl(items):
    return "\n".join(
        json.dumps({"messages": messages_for(item)}) for item in items
    )

def write_jsonl(items, filename):
    Path(filename).parent.mkdir(parents=True, exist_ok=True)
    Path(filename).write_text(make_jsonl(items), encoding="utf-8")

In [6]:
# Use 100 train + 50 val (OpenAI recommends 50–100)

fine_tune_train = train[:100]
fine_tune_val = val[:50]

write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")
write_jsonl(fine_tune_val, "jsonl/fine_tune_validation.jsonl")

print("JSONL files written.")

JSONL files written.


## Upload and fine-tune (optional)

Uncomment to upload files and start a fine-tuning job. Costs apply.

In [7]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    val_file = openai.files.create(file=f, purpose="fine-tune")
job = openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=val_file.id,
    model="gpt-4.1-nano-2025-04-14",
    suffix="pricer",
    hyperparameters={       
        "n_epochs": 1,
        "batch_size": 1,
    }
)

In [8]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id


In [9]:
job_id


'ftjob-z7B9y6udQJPn04pauDHCDrU3'

In [24]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-z7B9y6udQJPn04pauDHCDrU3', created_at=1773060675, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-nano-2025-04-14:kimanij:pricer:DHUVh2mW', finished_at=1773061229, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-cQQgB3gRiOg7A0VvVChJiZSF', result_files=['file-4ZDEAjv4NrT3nHN33KSkDN'], seed=444222428, status='succeeded', trained_tokens=11290, training_file='file-T3G1JYUd4SjPV6oZxCMREW', validation_file='file-RoAGrpAgmUXzSQ2pVcAVYE', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_e

In [23]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-QLOFVv0bBRFxrJrHvYVOJRty', created_at=1773061956, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-G7FdFFwj4sn088OqychhrqLt', created_at=1773061954, level='info', message='Usage policy evaluations completed, model is now enabled for sampling', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-XtJd13wI3iWxtRVDJoZXt3Xs', created_at=1773061954, level='info', message='Moderation checks for snapshot ft:gpt-4.1-nano-2025-04-14:kimanij:pricer:DHUVh2mW passed.', object='fine_tuning.job.event', data={'blocked': False, 'results': [{'flagged': False, 'category': 'harassment/threatening', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual/minors', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'propaganda', 'enforcement': 'blocking'

## Fine-tuned predictor

Replace `FINE_TUNED_MODEL` with your model ID after the job completes.

In [ ]:
FINE_TUNED_MODEL = "t:gpt-4.1-nano-2025-04-14:kimanij:pricer:DHUVh2mW"  # Replace with your model ID

def fine_tuned_pricer(item):
    prompt = f"Estimate the price of this product. Respond with the price, no explanation.\n\n{item.summary}"
    r = openai.chat.completions.create(
        model=FINE_TUNED_MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=50,
    )
    return r.choices[0].message.content.strip()

evaluate(fine_tuned_pricer, test, size=50)
